## Policy-Bound Decision System for Customer Support

The AI must:

    Answer only when context is sufficient

    Refuse when critical context is missing

    Ask clarification questions instead of guessing


## Why This Task Matters

In real systems:
- Context is often incomplete
- User data may be missing or conflicting
- AI must NOT guess or hallucinate

This task focuses on **controlled failure**, not better answers.

## Step 1: Add Guardrails to Static Context

We now enforce *when the AI should refuse to answer*.

In [1]:
SYSTEM_CONTEXT_GUARDED = """
You are a customer support assistant for an EdTech platform.

Rules:
- Do not assume missing information
- If required data is missing, ask a clarification question
- If policy cannot be applied, respond with "Unable to determine"
- Never hallucinate eligibility
- Be polite and professional
"""

## Step 2: Incomplete User Context (Simulated Real-World Issue)

Here, we intentionally remove critical information.

In [2]:
user_query = "Can I get a refund?"

user_profile_incomplete = {
    "role": "student"
    # Missing purchase date
    # Missing course progress
}

## Step 3: Assemble Context with Missing Information

Notice: we DO NOT fill missing values.

In [3]:
REFUND_POLICY = """
Refund Policy:
- Refunds are allowed only within 7 days of purchase
- Course progress must be below 20%
- Subscriptions are non-refundable
"""

In [4]:
final_prompt_incomplete = f"""
{SYSTEM_CONTEXT_GUARDED}

Company Policy:
{REFUND_POLICY}

User Profile:
- Role: {user_profile_incomplete['role']}

User Question:
{user_query}
"""

In [14]:
from google.colab import userdata
from openai import OpenAI

# 1. Load API key from Colab Secrets
MY_API_KEY = userdata.get("api_key")

# 2. Create OpenAI-compatible client
client = OpenAI(
    api_key=MY_API_KEY,
    base_url="https://nexusapi.navigatelabsai.com"
)

# 3. Send request to the model
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_CONTEXT_GUARDED
        },
        {
            "role": "user",
            "content": final_prompt_incomplete
        }
    ]
)

# 4. Print model response
print(response.choices[0].message.content)

I can help you with that! To determine if you're eligible for a refund, I'll need a bit more information.

Could you please tell me:
1.  When did you purchase the course or subscription?
2.  What is the name of the course or subscription you are requesting a refund for?
3.  What is your current progress in the course (e.g., 10%, 30%)?

Once I have this information, I can check it against our refund policy.


## What Should Happen?

Correct behavior:
- AI asks for missing information
OR
- AI says it cannot determine eligibility

This is **success**, not failure.

## Step 4: Conflicting Context

Real systems often receive contradictory data.

In [15]:
user_profile_conflict = {
    "role": "student",
    "course_progress": "10%",
    "purchase_days_ago": 30  # Conflicts with refund policy
}

In [16]:
final_prompt_conflict = f"""
{SYSTEM_CONTEXT_GUARDED}

Company Policy:
{REFUND_POLICY}

User Profile:
- Role: {user_profile_conflict['role']}
- Course Progress: {user_profile_conflict['course_progress']}
- Purchased: {user_profile_conflict['purchase_days_ago']} days ago

User Question:
{user_query}
"""

In [17]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content": SYSTEM_CONTEXT_GUARDED},
        {"role": "user", "content": final_prompt_conflict}
    ]
)

print(response.choices[0].message.content)

Thank you for reaching out!

Based on our refund policy, refunds are allowed only within 7 days of purchase. Your purchase was made 30 days ago, which unfortunately falls outside of this 7-day window.

Therefore, we are unable to process a refund at this time.


## Why This Is Important

The AI:
- Followed policy
- Did not hallucinate eligibility
- Handled conflicting data correctly

This is how **production AI systems** behave.

## Your Task (Hands-On)

Choose your domain and implement:
1. One missing-context scenario
2. One conflicting-context scenario
3. Guardrails that prevent guessing

You must show:
- AI refusal
- AI clarification
- AI safe fallback